# Alpha-shape plaque-edge proximity workflow

Output-free GitHub copy of the latest local alphashape workflow. Generated tables, figures, and H5AD files should stay outside version control.


## 1. Load the data

In [ ]:
from pathlib import Path

In [ ]:
INFECTED_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/"
    "MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/"
    "Spatial transcriptomics datasets/20240627__192310__KAECH_AD_GBM_240627/"
    "output-XETG00224__0023902__APPPS1_infected_453f30__20240627__192344"
)

MOCK_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/"
    "MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/"
    "Spatial transcriptomics datasets/20240627__192310__KAECH_AD_GBM_240627/"
    "output-XETG00224__0023902__APPPS1_mock_437f2__20240627__192344"
)

IF_PATH = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/"
    "MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/"
    "Spatial transcriptomics datasets/20240627__192310__KAECH_AD_GBM_240627/"
    "IF after Xenium run/OME-TIFF images for Xenium Explorer/"
    "APPPS1_infectedIF.ome.tif"
)

for path in [INFECTED_DIR, MOCK_DIR, IF_PATH]:
    print(path.exists(), path)


In [ ]:
import numpy as np
import tifffile as tf
import anndata as ad
import scanpy as sc

In [ ]:
with tf.TiffFile(IF_PATH) as tif:
    # Read the full image volume stack
    image_stack = tif.asarray()
    # Read XML metadata if you need pixel size scaling dimensions
    metadata_xml = tif.ome_metadata

# ASSUMPTION: Identify which channel corresponds to your infected tissue mask.
# Replace '0' with the specific index channel containing your mask label/stain.
infected_tissue_mask = (image_stack[0] > 0).astype(bool) 
# If your input image is a single-channel mask file:
# infected_tissue_mask = (image_stack > 0).astype(bool)



## ----------------------------------------------------
## 2. LOAD AND TRANSFORM THE ANNDATA MATRIX
## ----------------------------------------------------

In [ ]:

adata = ad.read_h5ad('/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/05_cluster_annotations/06_manual_annotation_08062026.h5ad')
adata.raw = adata

# sc.pp.log1p(adata)
transformed_expression_matrix = adata.X
if isinstance(transformed_expression_matrix, np.ndarray) == False:
    transformed_expression_matrix = transformed_expression_matrix.toarray()


## ----------------------------------------------------
## 3. RUN SQUIDPY ANALYSIS BEFORE THE OVERLAY
## ----------------------------------------------------

In [ ]:
import squidpy as sq
adata.obsm["spatial"] = adata.obsm["X_spatial"].copy()
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=8)
sq.gr.nhood_enrichment(adata, cluster_key="leiden", seed=0)
sq.pl.nhood_enrichment(adata, cluster_key="leiden", method="ward", title=f"Infected: neighborhood enrichment")
sq.gr.spatial_autocorr(adata, mode="moran", genes=adata.var_names.to_list(), n_perms=100)

print(f"Infected : top spatial genes")
display(adata.uns["moranI"].head(20))



## ----------------------------------------------------
## 4. VERIFY AXIS PARITY / ALIGNMENT (CRITICAL STEP)
## ----------------------------------------------------
Check if spatial layout array matches your OME-TIFF dimension ordering
Image matrices load as (Y, X) while Spatial coordinates usually export as (X, Y)

In [ ]:
from skimage import exposure
import matplotlib.pyplot as plt

In [ ]:
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm["spatial"]
    # Confirm shape bounds before overlay operations
    img_height, img_width = infected_tissue_mask.shape
    max_x, max_y = np.max(spatial_coords, axis=0)
    if max_x > img_width or max_y > img_height:
        print("⚠️ Warning: Spatial coordinates exceed image boundaries.")
        print("Consider transposing matrix or checking orientation scales!")

In [ ]:
pixel_size_um = 0.2125

xenium_pixels = spatial_coords / pixel_size_um
x_pixels = xenium_pixels[:, 0]
y_pixels = xenium_pixels[:, 1]


## -------------------------------------------------------------------------
## 4. CHOOSE IMAGE CHANNELS & OPTIMIZE CONTRAST
## -------------------------------------------------------------------------
Channel 0 = Plaque/Infected Marker
Adjust contrast using adaptive histogram equalization for clean presentation

In [ ]:

ch_plaque = exposure.equalize_adapthist(image_stack[0], clip_limit=0.03)
ch_nuclei = exposure.equalize_adapthist(image_stack[2], clip_limit=0.03)

# Build a composite RGB visualization array matching image pixel dimensions
ny, nx = image_stack.shape[1], image_stack.shape[2]
rgb_composite = np.zeros((ny, nx, 3), dtype=np.float32)

rgb_composite[..., 0] = ch_plaque  # Red channel: Plaques/Infection
rgb_composite[..., 2] = ch_nuclei  # Blue channel: Nuclei

In [ ]:
def read_ome_level(path, level=-1):
    with tf.TiffFile(path) as tif:
        series = tif.series[0]
        levels = list( getattr(series, "levels", None) or [series])
        if level < 0:
            level = len(levels) + level
        level = max(0, min(level, len(levels) - 1),)
        image = levels[level].asarray()
        information = {
            "axes": levels[level].axes,
            "level": level,
            "level_shape": levels[level].shape,
            "full_shape": levels[0].shape,
            "number_of_levels": len(levels),
        }
    return image, information

infected_morphology, infected_morphology_info = (read_ome_level(INFECTED_DIR / "morphology.ome.tif", level=-1))

mock_morphology, mock_morphology_info = (read_ome_level(MOCK_DIR / "morphology.ome.tif", level=-1))

# Maximum projection over the Xenium DAPI Z-stack.
infected_dapi = infected_morphology.max(axis=0)
mock_dapi = mock_morphology.max(axis=0)

print("Infected:", infected_morphology_info)
print("Mock:", mock_morphology_info)

## -------------------------------------------------------------------------
## 5. OVERLAY CELLS ON NATIVE XENIUM IMAGES SEPARATELY
## -------------------------------------------------------------------------

In [ ]:
adata_infected = adata[adata.obs["batch"] == "AD_inf"].copy()
adata_mock = adata[adata.obs["batch"] == "AD_mock"].copy()

#### Load the alignment matrices

In [ ]:
XENIUM_PIXEL_SIZE_UM = 0.2125

INFECTED_ALIGNMENT_MATRIX_PATH = Path("/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/Spatial transcriptomics datasets/Post Xenium IF/APPPS1_infectedIF_alignment_files/matrix.csv")
MOCK_ALIGNMENT_MATRIX_PATH = Path("/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/Spatial transcriptomics datasets/Post Xenium IF/Morphology_alignment_files_APP_Mock/matrix.csv")

A_INFECTED_IF_TO_XENIUM = np.loadtxt(INFECTED_ALIGNMENT_MATRIX_PATH, delimiter=",")
A_MOCK_IF_TO_XENIUM = np.loadtxt(MOCK_ALIGNMENT_MATRIX_PATH, delimiter=",")

AFFINE_MATRICES = {
    "infected": A_INFECTED_IF_TO_XENIUM,
    "mock": A_MOCK_IF_TO_XENIUM,
}

print("Infected matrix:")
print(A_INFECTED_IF_TO_XENIUM)

print("Mock matrix:")
print(A_MOCK_IF_TO_XENIUM)

In [ ]:
if_thumbnail, if_information = read_ome_level(IF_PATH, level=-1)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for channel, ax in enumerate(axes.flat):
    image = if_thumbnail[channel].astype(float)
    low, high = np.percentile(image, [1, 99.8])
    displayed_image = exposure.rescale_intensity(image, in_range=(low, high))
    ax.imshow(displayed_image, cmap="gray")
    ax.set_title(f"IF channel {channel}")
    ax.axis("off")

plt.tight_layout()
plt.show()


#### Load the plaque channel 0

In [ ]:
PLAQUE_CHANNEL = 0
PLAQUE_LEVEL = 1

def apply_affine_xy(xy, affine_matrix):
    xy = np.asarray(xy, dtype=float)
    homogeneous_xy = np.column_stack([xy, np.ones(len(xy))])
    print(affine_matrix)
    transformed = homogeneous_xy @ affine_matrix.T
    return transformed[:, :2] / transformed[:, 2:3]

def cells_to_if_level(adata_sample, affine_if_to_xenium, image_information):
    cell_xenium_pixels = adata_sample.obsm["spatial"] / XENIUM_PIXEL_SIZE_UM
    cell_if_full_pixels = apply_affine_xy(cell_xenium_pixels, np.linalg.inv(affine_if_to_xenium))

    full_y, full_x = image_information["full_shape"][-2:]
    level_y, level_x = image_information["level_shape"][-2:]
    downsample_xy = np.array([full_x / level_x, full_y / level_y])

    cell_if_level_pixels = cell_if_full_pixels / downsample_xy
    return cell_if_level_pixels, downsample_xy

if_plaque_image, if_plaque_information = read_ome_level(IF_PATH, level=1)
plaque_image = if_plaque_image[0].astype(np.float32)



#### Transform combined cells into IF coordinates

In [ ]:
infected_cell_if_xy, downsample_xy = cells_to_if_level(adata_infected, A_INFECTED_IF_TO_XENIUM, if_plaque_information)

In [ ]:
#### Identify valid cells separately

def valid_image_coordinates(xy, image_shape):
    return (
        np.isfinite(xy).all(axis=1)
        & (xy[:, 0] >= 0)
        & (xy[:, 1] >= 0)
        & (xy[:, 0] < image_shape[1])
        & (xy[:, 1] < image_shape[0])
    )

infected_valid = valid_image_coordinates(infected_cell_if_xy, plaque_image.shape)
infected_cells_if_valid = infected_cell_if_xy[infected_valid]

print("Valid infected cells:", infected_valid.sum())


In [ ]:
def plot_separate_if_overlay(image, cell_xy, sample_name, color, padding=100, point_size=0.2):
    x_min = max(0, int(np.floor(cell_xy[:, 0].min())) - padding)
    x_max = min(image.shape[1], int(np.ceil(cell_xy[:, 0].max())) + padding)
    y_min = max(0, int(np.floor(cell_xy[:, 1].min())) - padding)
    y_max = min(image.shape[0], int(np.ceil(cell_xy[:, 1].max())) + padding)

    image_crop = image[y_min:y_max, x_min:x_max]
    cell_crop_xy = cell_xy - np.array([x_min, y_min])

    display_values = image_crop[np.isfinite(image_crop)]
    display_low, display_high = np.percentile(display_values, [1, 99.8])

    fig, ax = plt.subplots(figsize=(11, 9))
    ax.imshow(image_crop, cmap="gray", vmin=display_low, vmax=display_high)
    ax.scatter(
        cell_crop_xy[:, 0],
        cell_crop_xy[:, 1],
        s=point_size,
        c=color,
        alpha=0.35,
        linewidths=0,
        rasterized=True,
    )

    ax.set_title(f"{sample_name}: Xenium cells over post-Xenium IF")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    return {
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
    }
OVERLAY_CHANNEL = 0
overlay_image = if_plaque_image[OVERLAY_CHANNEL].astype(float)

infected_crop_information = plot_separate_if_overlay(
    image=overlay_image,
    cell_xy=infected_cells_if_valid,
    sample_name="Infected",
    color="lime",
    padding=100,
    point_size=0.2)


## Identify plaques

In [ ]:
from scipy.spatial import ConvexHull
from skimage.draw import polygon2mask
from skimage import morphology

infected_full = ConvexHull(infected_cells_if_valid)
infected_hull_xy = infected_cells_if_valid[infected_full.vertices]

infected_tissue_mask = polygon2mask(overlay_image.shape, infected_hull_xy[:, [1, 0]])

infected_tissue_mask = morphology.binary_erosion(infected_tissue_mask, morphology.disk(2))
print("Pixels inside infected tissue:", infected_tissue_mask.sum())


In [ ]:
display_values = overlay_image[infected_tissue_mask]
display_low, display_high = np.percentile(display_values, [1, 99.9])

plt.figure(figsize=(12, 10))
plt.imshow(overlay_image, cmap="gray", vmin=display_low, vmax=display_high)
plt.contour(infected_tissue_mask, levels=[0.5], colors="cyan", linewidths=0.8)
plt.scatter(infected_cells_if_valid[:, 0], infected_cells_if_valid[:, 1], s=0.1, c="lime", alpha=0.15)
plt.title("Infected tissue used for plaque detection")
plt.axis("off")
plt.tight_layout()
plt.show()

### Calculate the IF-level pixel size

In [ ]:
downsample_x, downsample_y = downsample_xy
affine_linear = A_INFECTED_IF_TO_XENIUM[:2, :2]

level_pixel_size_x_um = np.linalg.norm(affine_linear[:, 0]) * downsample_x * XENIUM_PIXEL_SIZE_UM
level_pixel_size_y_um = np.linalg.norm(affine_linear[:, 1]) * downsample_y * XENIUM_PIXEL_SIZE_UM
pixel_size_um = np.mean([level_pixel_size_x_um, level_pixel_size_y_um])
pixel_area_um2 = level_pixel_size_x_um * level_pixel_size_y_um

print("IF-level pixel size:", round(pixel_size_um, 3), "µm")
print("IF-level pixel area:", round(pixel_area_um2, 3), "µm²")

In [ ]:
from scipy import ndimage as ndi
from skimage import exposure

plaque_image = overlay_image.astype(np.float32)
plaque_image[~infected_tissue_mask] = 0
### Assuming that the background spots are lesser than 1 um and the plaque size is lesser than 50 um (before it was 25, for a more linient consideration, increasing to 50)
small_sigma_pixels = max(0.5, 1 / pixel_size_um)
background_sigma_pixels = max(3, 50 / pixel_size_um)

smoothed_image = ndi.gaussian_filter(plaque_image, sigma=small_sigma_pixels)
broad_background = ndi.gaussian_filter(plaque_image, sigma=background_sigma_pixels)

compact_signal = np.clip(smoothed_image - broad_background, 0, None)
compact_signal[~infected_tissue_mask] = 0

positive_signal = compact_signal[infected_tissue_mask]
normalization_value = np.percentile(positive_signal, 99.9)
normalized_image = np.clip(compact_signal / normalization_value, 0, 1 )
normalized_image[~infected_tissue_mask] = 0


#### Compare multiple percentile values for normalization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, percentile in zip(axes, [99, 99.5, 99.9]):
    normalization_value = np.percentile(positive_signal, percentile)

    test_image = np.clip(compact_signal / normalization_value, 0, 1)
    test_image[~infected_tissue_mask] = 0

    ax.imshow(test_image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"{percentile}th percentile\n"
        f"normalization = {normalization_value:.1f}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
spot_values = normalized_image[infected_tissue_mask]
display_high = np.percentile(spot_values, 99.9)

plt.figure(figsize=(12, 10))
plt.imshow(normalized_image, cmap="gray", vmin=0, vmax=display_high)
plt.title("Infected compact white-dot signal")
plt.axis("off")
plt.tight_layout()
plt.show()

### Find plaques using alphashape
An alpha shape does not detect plaques directly from a grayscale image. First you must identify plaque-positive pixels. Then:
1. Threshold the fluorescence signal.
2. Remove small objects.
3. Optionally separate touching objects with watershed.
4. Construct an alpha shape around each positive region.
5. Measure area and boundary directly from that alpha shape.

In [ ]:
import alphashape
import numpy as np
from scipy import ndimage as ndi
from skimage import filters, measure, morphology, segmentation, feature
from skimage.draw import polygon as rasterize_polygon
from shapely.geometry import Polygon, MultiPolygon

In [ ]:
MIN_PLAQUE_AREA_UM2 = 20
MAX_PLAQUE_AREA_UM2 = 5000

# Alpha shapes retain Delaunay triangles whose cirum radius is approximately no larger than this value
ALPHA_RADIUS_UM = 20

# Set True only is adjacent plaques are merging 
SPLIT_TOUCHING_PLAQUES = True

# Control watershed splitting, not plaque shape
WATERSHED_MIN_SEPARATION_UM = 5

MAX_BOUNDARY_POINTS = 2500


##### ------------------------------------------------------------
##### 1. Threshold the background-corrected image
##### ------------------------------------------------------------ 

In [ ]:
### Four intensity classes:
### background, weak, intermediate and strong
signal_values = normalized_image[infected_tissue_mask]
signal_values = signal_values[np.isfinite(signal_values) & (signal_values > 0)]
cutoffs = filters.threshold_multiotsu(signal_values, classes=4)
PLAQUE_THRESHOLD = cutoffs[1]

plaque_positive_mask = (normalized_image >= PLAQUE_THRESHOLD) & infected_tissue_mask
pixel_area_um2 = pixel_size_um**2
minimum_area_pixels = max(1, int(np.ceil(MIN_PLAQUE_AREA_UM2 / pixel_area_um2)))

plaque_positive_mask = morphology.binary_opening(plaque_positive_mask, morphology.disk(1))
plaque_positive_mask = morphology.binary_closing(plaque_positive_mask, morphology.disk(2))
plaque_positive_mask = morphology.remove_small_objects(plaque_positive_mask, min_size = minimum_area_pixels)
plaque_positive_mask = ndi.binary_fill_holes(plaque_positive_mask)

print("Multi-Otsu cutoffs:", cutoffs)
print("Plaque threshold:", PLAQUE_THRESHOLD)
print("Positive pixels:", plaque_positive_mask.sum())


#### Optionally separate touching plaques


In [ ]:
if SPLIT_TOUCHING_PLAQUES:
    distance_image = ndi.distance_transform_edt(plaque_positive_mask)
    minimum_separation_pixels = max(1, round(WATERSHED_MIN_SEPARATION_UM / pixel_size_um))
    peak_coordinates = feature.peak_local_max(distance_image, labels =plaque_positive_mask.astype(np.uint8), 
                                              min_distance=minimum_separation_pixels, exclude_border=False)
    markers = np.zeros(plaque_positive_mask.shape, dtype=np.int32)
    markers[peak_coordinates[:, 0], peak_coordinates[:, 1]] = np.arange(1, len(peak_coordinates) + 1)
    candidate_labels = segmentation.watershed(-distance_image, markers=markers, mask=plaque_positive_mask, watershed_line=True)
else:
    candidate_labels = measure.label(plaque_positive_mask)

print("Candidate plaque regions: ", candidate_labels.max())

In [ ]:
### Convert each region to an alpha shape

def extract_boundary_point(region_mask, max_points=2500):
    """Return region boundary coordinates in x, y order."""
    contours = measure.find_contours(region_mask.astype(float), level=0.5)
    points_xy = np.concatenate([contour[:, [1, 0]]for contour in contours], axis=0)
    if len(points_xy) > max_points:
        selected = np.linspace(0, len(points_xy) - 1, max_points, dtype=int)
        points_xy = points_xy[selected]
    return points_xy

In [ ]:
def polygon_parts(geometry):
    """Return the polygon components of a Shapely geometry."""
    if geometry.is_empty:
        []
    if isinstance(geometry, Polygon):
        return [geometry]
    if isinstance(geometry, MultiPolygon):
        return list(geometry.geoms)
    if hasattr(geometry, "geoms"):
        return [part for part in geometry.geoms if isinstance(part, Polygon)]
    return []

def rasterize_shapely_geometry(geometry, output_shape, offset_xy=(0, 0)):
    """Convert a Shapely Polygon or MultiPolygon to a mask."""
    output = np.zeros(output_shape, dtype=bool)
    offset_x, offset_y = offset_xy

    for part in polygon_parts(geometry):
        exterior_xy = np.asarray(part.exterior.coords)

        rr, cc = rasterize_polygon(
            exterior_xy[:, 1] + offset_y,
            exterior_xy[:, 0] + offset_x,
            shape=output_shape,
        )
        output[rr, cc] = True

        # Remove polygon holes.
        for interior in part.interiors:
            interior_xy = np.asarray(interior.coords)

            rr, cc = rasterize_polygon(
                interior_xy[:, 1] + offset_y,
                interior_xy[:, 0] + offset_x,
                shape=output_shape,
            )
            output[rr, cc] = False

    return output


In [ ]:
# alphashape uses inverse-distance units.
# A smaller alpha approaches the convex hull.
# A larger alpha produces tighter, more concave boundaries.
import pandas as pd
alpha_value = pixel_size_um / ALPHA_RADIUS_UM
alpha_labels = np.zeros(candidate_labels.shape, dtype=np.int32)
plaque_records = []
next_plaque_id = 1

for candidate_id in range(1, candidate_labels.max() + 1):
    candidate_mask = candidate_labels == candidate_id
    coordinate_yx = np.column_stack(np.nonzero(candidate_mask))
    if len(coordinate_yx) < minimum_area_pixels:
        continue
    y_min, x_min = coordinate_yx.min(axis=0)
    y_max, x_max = coordinate_yx.max(axis=0) + 1
    padding = 3
    y_min = max(0, y_min - padding)
    x_min = max(0, x_min - padding)
    y_max = min(candidate_mask.shape[0], y_max + padding)
    x_max = min(candidate_mask.shape[1], x_max + padding)

    local_region = candidate_mask[y_min:y_max, x_min:x_max]
    boundary_xy = extract_boundary_point(local_region, max_points=MAX_BOUNDARY_POINTS)
    if len(boundary_xy) < 4:
        continue
    geometry = alphashape.alphashape(boundary_xy, alpha_value)
    parts = polygon_parts(geometry)
    if not parts:
        continue
    geometry = max(parts, key=lambda part: part.area)
    local_alpha_mask = rasterize_shapely_geometry(geometry, output_shape=local_region.shape)
    local_alpha_masl = ndi.binary_fill_holes(local_alpha_mask)

    area_pixels = int(local_alpha_mask.sum())
    area_um2 = area_pixels * pixel_area_um2

    if not (MIN_PLAQUE_AREA_UM2 <= area_um2 <= MAX_PLAQUE_AREA_UM2):
        continue

    global_alpha_mask = np.zeros_like(candidate_mask, dtype=bool)
    global_alpha_mask[y_min:y_max, x_min:x_max] = local_alpha_mask
    # Keep the alpha shape assosciated with its original candidate
    # A one-pixel dilation allows slight boundary smoothing
    permitted_region = morphology.binary_dilation(candidate_mask, morphology.disk(1))
    global_alpha_mask &= permitted_region
    if not global_alpha_mask.any():
        continue
    alpha_labels[global_alpha_mask] = next_plaque_id

    props = measure.regionprops(global_alpha_mask.astype(np.uint8), intensity_image=plaque_image)[0]
    centroid_y, centroid_x = props.centroid
    measured_area_um2 = props.area * pixel_area_um2

    # Equivalent diameter is only a convenient area summary 
    # It is not used to assume or construct a circular plaque
    equivalent_diameter_um = 2 * np.sqrt(measured_area_um2 / np.pi)
    plaque_records.append(
        {
            "plaque_id": next_plaque_id,
            "candidate_id": candidate_id,
            "if_x": centroid_x,
            "if_y": centroid_y,
            "area_pixels": props.area,
            "area_um2": measured_area_um2,
            "equivalent_diameter_um": equivalent_diameter_um,
            "feret_diameter_um": (props.feret_diameter_max * pixel_size_um),
            "perimeter_um": (props.perimeter * pixel_size_um),
            "mean_intensity": props.mean_intensity,
            "max_intensity": plaque_image[global_alpha_mask].max()}
    )
    next_plaque_id += 1

alpha_plaque_mask = alpha_labels > 0
alpha_plaques = pd.DataFrame(plaque_records)

print("Alpha-shape plaques:", len(alpha_plaques))
display(alpha_plaques.head())


#### Plot the irregular boundaries

In [ ]:
display_values = plaque_image[infected_tissue_mask]
display_values = display_values[np.isfinite(display_values)]

display_low, display_high = np.percentile(display_values, [1, 99.9])

fig, ax = plt.subplots(figsize=(14, 12))

ax.imshow(plaque_image, cmap="gray", vmin=display_low, vmax=display_high)

ax.contour(alpha_plaque_mask, levels=[0.5], colors="lime", linewidths=0.8)

if not alpha_plaques.empty:
    ax.scatter(alpha_plaques["if_x"], alpha_plaques["if_y"], s=8, c="cyan")

ax.set_title(f"Alpha-shape plaque boundaries: "
            f"{len(alpha_plaques)} plaques")
ax.axis("off")

plt.tight_layout()
plt.show()

### 6. Find cells near the detected plaques
Distances are measured from the **circumference of each alpha-shape plaque**
(`alpha_labels`), not from the plaque centroid, so an irregular plaque keeps its
real outline.

For every cell:
1. Take the plaque boundary (contour) of every alpha shape and resample it to a
   sub-pixel point cloud.
2. Query the nearest boundary point for each cell centroid (`cKDTree`), in µm.
3. Give cells whose centroid lies **inside** an alpha shape a negative distance.
4. Bin the signed distance into the 15 / 20 / 30 / 40 µm bands.

Each cell is assigned to its single nearest plaque, so no cell is counted twice
when plaques sit closer together than 40 µm.

In [ ]:
# Distance bands measured outward from the plaque circumference (µm).
BAND_EDGES_UM = (15, 20, 30, 40)

# Cell-type column used for the per-band composition tables.
CELLTYPE_KEY = "manual_celltype"

# Plaque boundaries are resampled to at most this spacing, in IF-level pixels,
# so that a nearest-point query approximates the true polygon distance.
BOUNDARY_SPACING_PX = 1.0

PROXIMITY_OUTPUT_DIR = Path("plaque_results") / "alpha_edge_proximity"
PROXIMITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Band edges (µm):", BAND_EDGES_UM)
print("IF-level pixel size:", round(pixel_size_um, 4), "µm")
print("Plaques available:", int(alpha_labels.max()))

### Extract the circumference of every alpha-shape plaque

In [ ]:
from scipy.spatial import cKDTree

def densify_polyline(points_xy, max_spacing=1.0):
    """Insert points so consecutive vertices are at most max_spacing apart."""
    points_xy = np.asarray(points_xy, dtype=float)
    if len(points_xy) < 2:
        return points_xy

    segments = np.diff(points_xy, axis=0)
    lengths = np.hypot(segments[:, 0], segments[:, 1])

    densified = [points_xy[:1]]
    for index, length in enumerate(lengths):
        steps = max(1, int(np.ceil(length / max_spacing)))
        fractions = (np.arange(1, steps + 1) / steps)[:, None]
        densified.append(points_xy[index] + fractions * segments[index])

    return np.vstack(densified)


def plaque_boundary_points(label_image, max_spacing=1.0):
    """Circumference coordinates of every labelled plaque.

    Returns boundary coordinates as x, y in IF-level pixels together with the
    plaque_id each boundary point belongs to. Interior holes of an alpha shape
    are returned as well, so a cell sitting in a hole is measured against the
    boundary it is actually closest to.
    """
    boundary_xy = []
    boundary_ids = []

    object_slices = ndi.find_objects(label_image)

    for index, slices in enumerate(object_slices):
        plaque_id = index + 1
        if slices is None:
            continue

        # Pad by one pixel so plaques touching the crop edge still close.
        region_mask = np.pad((label_image[slices] == plaque_id).astype(float), 1)
        x_offset = slices[1].start - 1
        y_offset = slices[0].start - 1

        for contour in measure.find_contours(region_mask, level=0.5):
            # find_contours returns row, col -> convert to global x, y.
            contour_xy = np.column_stack(
                [contour[:, 1] + x_offset, contour[:, 0] + y_offset]
            )
            contour_xy = densify_polyline(contour_xy, max_spacing=max_spacing)
            boundary_xy.append(contour_xy)
            boundary_ids.append(np.full(len(contour_xy), plaque_id, dtype=np.int32))

    if not boundary_xy:
        raise ValueError("No plaque boundaries were found in alpha_labels.")

    return np.vstack(boundary_xy), np.concatenate(boundary_ids)

boundary_xy_pixels, boundary_plaque_ids = plaque_boundary_points(
    alpha_labels, max_spacing=BOUNDARY_SPACING_PX
)

print("Boundary points:", len(boundary_xy_pixels))
print("Plaques with a boundary:", len(np.unique(boundary_plaque_ids)))


### Signed distance from each cell to the nearest plaque circumference

In [ ]:
# Query in micrometres so the bands are physical distances.
boundary_tree = cKDTree(boundary_xy_pixels * pixel_size_um)

cell_xy_pixels = infected_cell_if_xy
query_xy_um = np.where(
    infected_valid[:, None], cell_xy_pixels, 0.0
) * pixel_size_um

edge_distance_um, nearest_boundary_index = boundary_tree.query(
    query_xy_um, k=1, workers=-1
)
nearest_plaque_id = boundary_plaque_ids[nearest_boundary_index].astype(float)

# A centroid that falls inside an alpha shape gets a negative distance.
image_height, image_width = alpha_labels.shape
sample_column = np.clip(np.rint(cell_xy_pixels[:, 0]), 0, image_width - 1).astype(int)
sample_row = np.clip(np.rint(cell_xy_pixels[:, 1]), 0, image_height - 1).astype(int)
label_under_cell = alpha_labels[sample_row, sample_column]

inside_plaque = infected_valid & (label_under_cell > 0)
signed_edge_distance_um = np.where(inside_plaque, -edge_distance_um, edge_distance_um).astype(float)
nearest_plaque_id = np.where(inside_plaque, label_under_cell, nearest_plaque_id)

# Cells whose IF coordinates fall outside the image cannot be scored.
signed_edge_distance_um[~infected_valid] = np.nan
nearest_plaque_id[~infected_valid] = np.nan

print("Cells scored:", int(infected_valid.sum()), "of", len(cell_xy_pixels))
print("Cells inside a plaque:", int(inside_plaque.sum()))
print(
    "Edge distance (µm) percentiles:",
    np.round(
        np.nanpercentile(signed_edge_distance_um, [0, 5, 25, 50, 75, 95, 100]), 2
    )
)
print("Cells scored:", int(infected_valid.sum()), "of", len(cell_xy_pixels))
print("Cells inside a plaque:", int(inside_plaque.sum()))
print("Edge distance (µm) range:", 
      np.round(np.nanpercentile(signed_edge_distance_um, [0, 5, 25, 40, 50, 75, 100]), 2))


### Assign each cell to a 15 / 20 / 30 / 40 µm band

In [ ]:
band_edges = [-np.inf, 0.0, *[float(edge) for edge in BAND_EDGES_UM], np.inf]
band_labels = ["inside_plaque", f"0-{BAND_EDGES_UM[0]}"]
band_labels += [
    f"{low}-{high}" for low, high in zip(BAND_EDGES_UM, BAND_EDGES_UM[1:])
]
band_labels += [f">{BAND_EDGES_UM[-1]}"]

plaque_edge_band = pd.cut(
    signed_edge_distance_um,
    bins=band_edges,
    labels=band_labels,
    right=True,
    include_lowest=True,
)

adata_infected.obs["plaque_edge_distance_um"] = signed_edge_distance_um
adata_infected.obs["nearest_plaque_id"] = nearest_plaque_id
adata_infected.obs["inside_plaque"] = inside_plaque
adata_infected.obs["plaque_proximity_valid"] = infected_valid
adata_infected.obs["plaque_edge_band_um"] = pd.Categorical(
    plaque_edge_band, categories=band_labels, ordered=True
)

print(plaque_edge_band.value_counts(dropna=False))

### Per-cell table, band counts and cell-type composition

In [ ]:
# Cumulative membership: within X µm of the circumference, plaque interior included.
# Unscored cells compare False against NaN, which is the behaviour we want.
for band_um in BAND_EDGES_UM:
    adata_infected.obs[f"within_{band_um}um_of_plaque"] = (
        signed_edge_distance_um <= band_um
    ) & infected_valid

PROXIMITY_COLUMNS = [
    "plaque_edge_distance_um",
    "nearest_plaque_id",
    "inside_plaque",
    "plaque_proximity_valid",
    "plaque_edge_band_um",
    *[f"within_{band_um}um_of_plaque" for band_um in BAND_EDGES_UM],
]

# Carry the annotation back onto the combined object (mock cells stay unscored).
for column in PROXIMITY_COLUMNS:
    adata.obs[column] = adata_infected.obs[column].reindex(adata.obs_names)

print(adata_infected.obs["plaque_edge_band_um"].value_counts().reindex(band_labels))
# display(band_summary)


In [ ]:
if CELLTYPE_KEY in proximity_table.columns:
    band_composition_counts = pd.crosstab(
        proximity_table["plaque_edge_band_um"],
        proximity_table[CELLTYPE_KEY],
        dropna=False,
    ).reindex(band_labels).fillna(0).astype(int)

    # Column-wise percentage answers "where does this cell type sit?".
    celltype_percent_by_band = 100 * band_composition_counts.div(
        band_composition_counts.sum(axis=0).replace(0, np.nan), axis=1
    )
    # Row-wise percentage answers "what is this band made of?".
    band_percent_by_celltype = 100 * band_composition_counts.div(
        band_composition_counts.sum(axis=1).replace(0, np.nan), axis=0
    )

    print("Cell counts per band and cell type")
    display(band_composition_counts)
    print("Composition of each band (row %)")
    display(band_percent_by_celltype.round(2))
else:
    print(f"Column {CELLTYPE_KEY!r} not in adata_infected.obs - skipping composition.")
    band_composition_counts = None
    celltype_percent_by_band = None
    band_percent_by_celltype = None

In [ ]:
proximity_table = pd.DataFrame(
    {
        "cell_id": adata_infected.obs_names.to_numpy(),
        "if_x": infected_cell_if_xy[:, 0],
        "if_y": infected_cell_if_xy[:, 1],
        "x_xenium_um": adata_infected.obsm["spatial"][:, 0],
        "y_xenium_um": adata_infected.obsm["spatial"][:, 1],
        "plaque_edge_distance_um": signed_edge_distance_um,
        "nearest_plaque_id": nearest_plaque_id,
        "inside_plaque": inside_plaque,
        "plaque_edge_band_um": adata_infected.obs["plaque_edge_band_um"].to_numpy(),
    }
)

if CELLTYPE_KEY in adata_infected.obs.columns:
    proximity_table[CELLTYPE_KEY] = adata_infected.obs[CELLTYPE_KEY].to_numpy()

proximity_table = proximity_table.loc[infected_valid].reset_index(drop=True)

# Exclusive bands.
band_counts = (
    proximity_table["plaque_edge_band_um"]
    .value_counts()
    .reindex(band_labels)
    .fillna(0)
    .astype(int)
)
band_summary = pd.DataFrame(
    {
        "summary_type": "exclusive_band",
        "group_um": band_labels,
        "cell_count": band_counts.to_numpy(),
    }
)

# Cumulative bands: everything within X µm of the circumference.
cumulative_rows = [
    {
        "summary_type": "cumulative",
        "group_um": f"within_{band_um}",
        "cell_count": int(
            (proximity_table["plaque_edge_distance_um"] <= band_um).sum()
        ),
    }
    for band_um in BAND_EDGES_UM
]

band_summary = pd.concat(
    [band_summary, pd.DataFrame(cumulative_rows)], ignore_index=True
)
band_summary["percent_of_cells"] = (
    100 * band_summary["cell_count"] / len(proximity_table)
)

print("Scored cells:", len(proximity_table))
display(band_summary)


In [ ]:
if CELLTYPE_KEY in proximity_table.columns:
    band_composition_counts = pd.crosstab(
        proximity_table["plaque_edge_band_um"],
        proximity_table[CELLTYPE_KEY],
        dropna=False,
    ).reindex(band_labels).fillna(0).astype(int)

    # Column-wise percentage answers "where does this cell type sit?".
    celltype_percent_by_band = 100 * band_composition_counts.div(
        band_composition_counts.sum(axis=0).replace(0, np.nan), axis=1
    )
    # Row-wise percentage answers "what is this band made of?".
    band_percent_by_celltype = 100 * band_composition_counts.div(
        band_composition_counts.sum(axis=1).replace(0, np.nan), axis=0
    )

    print("Cell counts per band and cell type")
    display(band_composition_counts)
    print("Composition of each band (row %)")
    display(band_percent_by_celltype.round(2))
else:
    print(f"Column {CELLTYPE_KEY!r} not in adata_infected.obs - skipping composition.")
    band_composition_counts = None
    celltype_percent_by_band = None
    band_percent_by_celltype = None


#### Cells per band for each individual plaque


In [ ]:
within_outer_band = proximity_table["plaque_edge_distance_um"] <= BAND_EDGES_UM[-1]

per_plaque_band_counts = (
    proximity_table.loc[within_outer_band]
    .groupby(["nearest_plaque_id", "plaque_edge_band_um"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=band_labels, fill_value=0)
)
per_plaque_band_counts.index = per_plaque_band_counts.index.astype(int)
per_plaque_band_counts = per_plaque_band_counts.drop(
    columns=[f">{BAND_EDGES_UM[-1]}"], errors="ignore"
)
per_plaque_band_counts["total_within_outer_band"] = per_plaque_band_counts.sum(axis=1)

per_plaque_summary = alpha_plaques.merge(
    per_plaque_band_counts,
    left_on="plaque_id",
    right_index=True,
    how="left",
).fillna({column: 0 for column in per_plaque_band_counts.columns})

display(
    per_plaque_summary[
        ["plaque_id", "area_um2", "equivalent_diameter_um", *per_plaque_band_counts.columns]
    ].head(20)
)


#### Plot the bands


In [ ]:
band_palette = plt.colormaps["Spectral"](np.linspace(0.02, 0.98, len(band_labels)))
band_colors = {label: band_palette[index] for index, label in enumerate(band_labels)}
band_colors[band_labels[0]] = np.array([0.45, 0.0, 0.05, 1.0])   # inside plaque
band_colors[band_labels[-1]] = np.array([0.80, 0.80, 0.80, 1.0])  # beyond the last band

fig, axes = plt.subplots(1, 2, figsize=(19, 8))

# Left: whole section, cells coloured by band.
ax = axes[0]
ax.imshow(plaque_image, cmap="gray", vmin=display_low, vmax=display_high)
for label in band_labels[::-1]:
    selected = proximity_table["plaque_edge_band_um"].astype(str) == label
    ax.scatter(
        proximity_table.loc[selected, "if_x"],
        proximity_table.loc[selected, "if_y"],
        s=0.5 if label == band_labels[-1] else 1.5,
        color=band_colors[label],
        alpha=0.25 if label == band_labels[-1] else 0.8,
        linewidths=0,
        label=f"{label} µm ({int(selected.sum())})",
        rasterized=True,
    )
ax.contour(alpha_plaque_mask, levels=[0.5], colors="white", linewidths=0.5)
ax.set_title("Cells by distance from the plaque circumference")
ax.axis("off")
ax.legend(markerscale=6, frameon=False, fontsize=8, loc="upper right")

# Right: histogram of edge distances.
ax = axes[1]
finite_distance = proximity_table["plaque_edge_distance_um"]
finite_distance = finite_distance[np.isfinite(finite_distance)]
ax.hist(
    finite_distance.clip(upper=2 * BAND_EDGES_UM[-1]),
    bins=120,
    color="#4575b4",
)
for band_um in BAND_EDGES_UM:
    ax.axvline(band_um, color="#d73027", linestyle="--", linewidth=1)
    ax.text(band_um, ax.get_ylim()[1] * 0.95, f" {band_um}", color="#d73027", fontsize=8)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Signed distance from plaque circumference (µm)")
ax.set_ylabel("Cells")
ax.set_title("Distance to the nearest plaque boundary")

plt.tight_layout()
plt.show()


In [ ]:
def plot_plaque_band_zoom(plaque_id, window_um=120, point_size=14):
    """Zoom on one plaque and draw the 15/20/30/40 µm rings around its outline."""
    plaque_row = alpha_plaques.loc[alpha_plaques["plaque_id"] == plaque_id].iloc[0]
    center_x, center_y = plaque_row["if_x"], plaque_row["if_y"]

    half_window_px = int(round((window_um + BAND_EDGES_UM[-1]) / pixel_size_um))
    x_min = max(0, int(center_x) - half_window_px)
    x_max = min(alpha_labels.shape[1], int(center_x) + half_window_px)
    y_min = max(0, int(center_y) - half_window_px)
    y_max = min(alpha_labels.shape[0], int(center_y) + half_window_px)

    crop_labels = alpha_labels[y_min:y_max, x_min:x_max]
    crop_image = plaque_image[y_min:y_max, x_min:x_max]

    # Distance to this plaque's outline only, computed on the crop.
    this_plaque = crop_labels == plaque_id
    crop_distance_um = ndi.distance_transform_edt(~this_plaque) * pixel_size_um

    in_crop = (
        proximity_table["if_x"].between(x_min, x_max)
        & proximity_table["if_y"].between(y_min, y_max)
    )
    crop_cells = proximity_table.loc[in_crop]

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(crop_image, cmap="gray", vmin=display_low, vmax=display_high)
    ax.contour(this_plaque, levels=[0.5], colors="white", linewidths=1.4)
    ax.contour(
        crop_distance_um,
        levels=list(BAND_EDGES_UM),
        colors=["#d73027", "#fc8d59", "#91cf60", "#4575b4"][: len(BAND_EDGES_UM)],
        linewidths=1.0,
    )

    for label in band_labels:
        selected = crop_cells["plaque_edge_band_um"].astype(str) == label
        ax.scatter(
            crop_cells.loc[selected, "if_x"] - x_min,
            crop_cells.loc[selected, "if_y"] - y_min,
            s=point_size,
            color=band_colors[label],
            edgecolors="black",
            linewidths=0.2,
            label=f"{label} µm",
        )

    ax.set_title(
        f"Plaque {plaque_id}: area {plaque_row['area_um2']:.0f} µm², "
        f"rings at {', '.join(str(edge) for edge in BAND_EDGES_UM)} µm"
    )
    ax.axis("off")
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    plt.tight_layout()
    plt.show()


if not alpha_plaques.empty:
    largest_plaques = (
        per_plaque_summary.sort_values("total_within_outer_band", ascending=False)
        .head(2)["plaque_id"]
        .astype(int)
        .tolist()
    )
    for plaque_id in largest_plaques:
        plot_plaque_band_zoom(plaque_id)


#### Save the proximity results


In [ ]:
proximity_table.to_csv(
    PROXIMITY_OUTPUT_DIR / "infected_cells_plaque_edge_distances.csv", index=False
)
band_summary.to_csv(
    PROXIMITY_OUTPUT_DIR / "plaque_edge_band_summary.csv", index=False
)
per_plaque_summary.to_csv(
    PROXIMITY_OUTPUT_DIR / "per_plaque_band_counts.csv", index=False
)

if band_composition_counts is not None:
    band_composition_counts.to_csv(
        PROXIMITY_OUTPUT_DIR / "band_celltype_counts.csv"
    )
    band_percent_by_celltype.round(4).to_csv(
        PROXIMITY_OUTPUT_DIR / "band_celltype_row_percent.csv"
    )
    celltype_percent_by_band.round(4).to_csv(
        PROXIMITY_OUTPUT_DIR / "celltype_percent_by_band.csv"
    )

# One file per cumulative distance cut-off.
for band_um in BAND_EDGES_UM:
    selected = proximity_table["plaque_edge_distance_um"] <= band_um
    proximity_table.loc[selected].to_csv(
        PROXIMITY_OUTPUT_DIR / f"infected_cells_within_{band_um}um.csv", index=False
    )
    print(f"within {band_um} µm:", int(selected.sum()), "cells")

print("Saved to:", PROXIMITY_OUTPUT_DIR.resolve())

In [ ]:
# Rebuilt here so this cell stands alone.
cutoff_labels = [f"<{edge}" for edge in BAND_EDGES_UM]
outer_label = f">={BAND_EDGES_UM[-1]}"
band_labels = [*cutoff_labels, outer_label]

INFECTED_BATCH = "AD_inf"
infected_mask = (adata.obs["batch"] == INFECTED_BATCH).to_numpy()

# adata_infected came from a boolean subset, so row order still matches.
# Checked positionally rather than by barcode: Xenium cell ids can repeat
# between the infected and mock sections.
assert infected_mask.sum() == adata_infected.n_obs, "infected cell count mismatch"
assert (adata.obs_names[infected_mask] == adata_infected.obs_names).all(), "row order mismatch between adata and adata_infected"


def scatter_to_full(values, fill=np.nan, dtype="float32"):
    """Place per-infected-cell values onto every row of the combined object."""
    full = np.full(adata.n_obs, fill, dtype=dtype)
    full[infected_mask] = values
    return full

# Signed distance to the plaque circumference. NaN = cell was never scored,
# which covers every mock cell and any infected cell off the IF image.
adata.obs["plaque_edge_distance_um"] = scatter_to_full(signed_edge_distance_um)
adata.obs["nearest_plaque_id"] = scatter_to_full(nearest_plaque_id)

# Was this cell scored at all? Tells "not near a plaque" apart from "no data".
adata.obs["plaque_proximity_valid"] = scatter_to_full(infected_valid, fill=False, dtype=bool)

adata.obs["inside_plaque"] = scatter_to_full(inside_plaque, fill=False, dtype=bool)

# Cumulative flags, rebuilt from the distance column so the dtype stays bool.
# NaN compares False, so unscored cells are False everywhere.
for band_um in BAND_EDGES_UM:
    adata.obs[f"within_{band_um}um_of_plaque"] = (adata.obs["plaque_edge_distance_um"] < band_um).to_numpy()

# Smallest cut-off each cell satisfies, as an ordered categorical. Derived
# straight from the distance column so this cell does not depend on a
# categorical having been written onto adata_infected.obs earlier.
adata.obs["plaque_edge_cutoff_um"] = pd.Categorical(
    pd.cut(
        adata.obs["plaque_edge_distance_um"].to_numpy(),
        bins=[-np.inf, *[float(edge) for edge in BAND_EDGES_UM], np.inf],
        labels=band_labels,
        right=False),
    categories=band_labels,
    ordered=True
)

# IF-level pixel coordinates, so the distances can be recomputed or replotted
# without redoing the affine transform.
if_pixel_xy = np.full((adata.n_obs, 2), np.nan, dtype="float32")
if_pixel_xy[infected_mask] = infected_cell_if_xy
adata.obsm["spatial_if_pixels"] = if_pixel_xy

# Keep adata_infected consistent with the combined object.
for column in [
    "plaque_edge_distance_um",
    "nearest_plaque_id",
    "plaque_proximity_valid",
    "inside_plaque",
    "plaque_edge_cutoff_um",
    *[f"within_{band_um}um_of_plaque" for band_um in BAND_EDGES_UM],
]:
    adata_infected.obs[column] = adata.obs[column].to_numpy()[infected_mask]

PROXIMITY_OBS_COLUMNS = [
    "plaque_edge_distance_um",
    "nearest_plaque_id",
    "plaque_proximity_valid",
    "inside_plaque",
    "plaque_edge_cutoff_um",
    *[f"within_{band_um}um_of_plaque" for band_um in BAND_EDGES_UM],
]

print(adata.obs[PROXIMITY_OBS_COLUMNS].dtypes)
print()
print("Scored cells in adata:", int(adata.obs["plaque_proximity_valid"].sum()))
print(adata.obs["plaque_edge_cutoff_um"].value_counts(dropna=False))


In [ ]:
# Tables that describe the plaques themselves; skipped if not in the kernel.
OPTIONAL_UNS_TABLES = {
    "plaque_table": "alpha_plaques",
    "plaque_cutoff_summary": "cutoff_summary",
    "plaque_per_plaque_counts": "per_plaque_summary",
}
for uns_key, variable_name in OPTIONAL_UNS_TABLES.items():
    table = globals().get(variable_name)
    if isinstance(table, pd.DataFrame):
        adata.uns[uns_key] = table.reset_index(drop=True)
    else:
        print(f"skipped uns[{uns_key!r}]: {variable_name} not defined")


In [ ]:
from datetime import datetime

adata.uns["plaque_proximity"] = {
    "method": "signed euclidean distance to the alpha-shape plaque circumference",
    "distance_sign": "negative inside a plaque, 0 on the boundary, positive outside",
    "cutoff_edges_um": np.asarray(BAND_EDGES_UM, dtype="float32"),
    "cutoff_comparison": "strict less-than",
    "cutoffs_are_cumulative": True,
    "scored_batch": INFECTED_BATCH,
    "if_level_pixel_size_um": float(pixel_size_um),
    "if_image_path": str(IF_PATH),
    "if_plaque_channel": int(PLAQUE_CHANNEL),
    "if_pyramid_level": int(PLAQUE_LEVEL),
    "boundary_spacing_px": float(BOUNDARY_SPACING_PX),
    "alpha_radius_um": float(ALPHA_RADIUS_UM),
    "plaque_threshold_normalized": float(PLAQUE_THRESHOLD),
    "min_plaque_area_um2": float(MIN_PLAQUE_AREA_UM2),
    "max_plaque_area_um2": float(MAX_PLAQUE_AREA_UM2),
    "split_touching_plaques": bool(SPLIT_TOUCHING_PLAQUES),
    "watershed_min_separation_um": float(WATERSHED_MIN_SEPARATION_UM),
    "n_plaques": int(len(alpha_plaques)),
    "n_cells_scored": int(len(proximity_table)),
    "created": datetime.now().strftime("%Y-%m-%d %H:%M"),
}


# Tables that describe the plaques themselves.
adata.uns["plaque_table"] = alpha_plaques.reset_index(drop=True)
# adata.uns["plaque_cutoff_summary"] = cutoff_summary.reset_index(drop=True)
adata.uns["plaque_per_plaque_counts"] = per_plaque_summary.reset_index(drop=True)

# The circumference point cloud, so distances can be recomputed from the h5ad
# alone without reloading the IF image.
adata.uns["plaque_boundary_xy_if_pixels"] = boundary_xy_pixels.astype("float32")
adata.uns["plaque_boundary_plaque_id"] = boundary_plaque_ids.astype("int32")

for key, value in adata.uns["plaque_proximity"].items():
    print(f"  {key}: {value}")


In [ ]:
H5AD_OUTPUT_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/"
    "RESULTS/20240627__192310__KAECH_AD_GBM_240627/08_Plaque_Proximity_Analysis"
)
H5AD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cutoff_tag = "_".join(str(band_um) for band_um in BAND_EDGES_UM)
PROXIMITY_H5AD_PATH = (
    H5AD_OUTPUT_DIR / f"adata_combined_with_alphashape_plaque_edge_{cutoff_tag}um.h5ad"
)

adata.write_h5ad(PROXIMITY_H5AD_PATH, compression="gzip")
print("Wrote", PROXIMITY_H5AD_PATH)
print("Size:", round(PROXIMITY_H5AD_PATH.stat().st_size / 1e6, 1), "MB")

# Read back and confirm nothing was silently dropped or retyped.
check = ad.read_h5ad(PROXIMITY_H5AD_PATH)
print()
print(check.obs[PROXIMITY_OBS_COLUMNS].dtypes)
print()
print(check.obs["plaque_edge_cutoff_um"].value_counts(dropna=False))
print()
for band_um in BAND_EDGES_UM:
    print(
        f"<{band_um} µm:", int(check.obs[f"within_{band_um}um_of_plaque"].sum()), "cells"
    )
print()
print("uns keys:", [key for key in check.uns if key.startswith("plaque")])
print("obsm keys:", list(check.obsm))
